In [ ]:
!pip install pypdf chromadb python-docx

In [ ]:
import os
import re
import uuid
from typing import List, Dict, Any

# Document parsing
from pypdf import PdfReader
import docx

# Embeddings + Vector DB
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb

# Google Gemini client
import google.generativeai as genai



In [ ]:
from config import GEMINI_API_KEY


In [ ]:
GEMINI_MODEL_NAME = "gemini-2.5-flash"

genai.configure(api_key=GEMINI_API_KEY)

# Embedding + reranker models
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
legal_model = SentenceTransformer('nlpaueb/legal-bert-base-uncased')
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Chroma DB client
chroma_client = chromadb.Client()
collection = chroma_client.create_collection("legal_docs")

InternalError: Collection [legal_docs] already exists

In [ ]:
def parse_pdf(path: str) -> str:
    reader = PdfReader(path)
    return "\n".join([page.extract_text() or "" for page in reader.pages])

def parse_docx(path: str) -> str:
    doc = docx.Document(path)
    return "\n".join([para.text for para in doc.paragraphs])

def parse_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def parse_document(path: str) -> str:
    if path.endswith(".pdf"):
        return parse_pdf(path)
    elif path.endswith(".docx"):
        return parse_docx(path)
    elif path.endswith(".txt"):
        return parse_text(path)
    else:
        raise ValueError("Unsupported file format. Use PDF, DOCX, or TXT.")

In [ ]:
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> List[str]:
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        end = min(len(words), start + chunk_size)
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks


In [ ]:
def ingest_document(path: str, collection):
    text = parse_document(path)   # parse once
    chunks = chunk_text(text)

    raw_chunks = []
    for chunk in chunks:
        if len(chunk.strip()) < 50:
            continue
        embedding = embedding_model.encode(chunk).tolist()
        collection.add(
            ids=[str(uuid.uuid4())],
            documents=[chunk],
            embeddings=[embedding],
        )
        raw_chunks.append(("Untitled Section", chunk))  # keep for summarizer
    return raw_chunks


In [ ]:
def split_by_headings(text: str) -> Dict[str, str]:
    """
    Split document text into sections based on headings.
    Headings are assumed to be lines in ALL CAPS or numbered like 1., 2.1, etc.
    """
    sections = {}
    current_heading = "Introduction"
    buffer = []

    lines = text.splitlines()
    for line in lines:
        line_stripped = line.strip()
        if re.match(r"^(\d+(\.\d+)*)\s", line_stripped) or line_stripped.isupper():
            if buffer:
                sections[current_heading] = "\n".join(buffer).strip()
                buffer = []
            current_heading = line_stripped
        else:
            buffer.append(line_stripped)

    # Add last section
    if buffer:
        sections[current_heading] = "\n".join(buffer).strip()

    return sections

In [ ]:
def simplify_summarize_section(title: str, text: str) -> str:
    """
    Summarize one section into bullet points (simplified, easy to understand).
    Skips if section is empty or trivial.
    """
    if not text.strip() or len(text.split()) < 20:  # skip too short sections
        return ""

    prompt = f"""
You are an assistant that explains documents in simple language.
Summarize the following section into 3–5 short, clear bullet points.

Rules:
- Use plain English (avoid jargon or legalese).
- Write as if explaining to someone with no legal/technical background.
- Keep sentences short and direct.
- If the section has no meaningful information, output "No important points."

Section Title: "{title}"

Section Text:
{text}

Output format:
- simple point 1
- simple point 2
...
"""
    model = genai.GenerativeModel("gemini-1.5-flash")
    response = model.generate_content(prompt).text
    return response

In [ ]:

def summarize_section(title: str, text: str) -> str:
    """
    Summarize one section into bullet points (if relevant).
    Skips if section is empty or trivial.
    """
    if not text.strip() or len(text.split()) < 20:  # skip too short sections
        return ""

    prompt = f"""
You are a legal assistant. Summarize the following section into 3–5 clear bullet points.
Do not invent a title, use the given one: "{title}".
If no meaningful points exist, return "No important points."

Section Text:
{text}

Output format:
- point 1
- point 2
...
"""
    model = genai.GenerativeModel("gemini-1.5-flash")
    response = model.generate_content(prompt).text
    return response


In [ ]:
def summarize_document(raw_chunks):
    summaries = {}
    for title, chunk in raw_chunks:
        summary = summarize_section(title, chunk)
        if summary and "No important points" not in summary:
            summaries[title] = summary
    return summaries


In [ ]:
def simplify_summarize_document(raw_chunks):
    summaries = {}
    for title, chunk in raw_chunks:
        summary = simplify_summarize_section(title, chunk)
        if summary and "No important points" not in summary:
            summaries[title] = summary
    return summaries


In [ ]:
def retrieve_and_rerank(query: str, top_k: int = 5) -> List[str]:
    query_emb = embedding_model.encode([query]).tolist()[0]

    results = collection.query(
        query_embeddings=[query_emb],
        n_results=top_k * 3  # get more, then rerank
    )

    candidates = results["documents"][0]
    scores = reranker.predict([(query, c) for c in candidates])

    reranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in reranked[:top_k]]


In [ ]:
def simplify_clause(clause_query: str, doc_type: str = "contract") -> str:
    retrieved_chunks = retrieve_and_rerank(clause_query)
    context = "\n".join(retrieved_chunks)

    prompt = f"""
You are a legal simplifier. The user provided a {doc_type}.
Clause/query: {clause_query}

Relevant context from the document:
{context}

Simplify this clause into plain English so a non-lawyer can understand.
"""
    model = genai.GenerativeModel(GEMINI_MODEL_NAME)
    response = model.generate_content(prompt)
    return response.text


In [ ]:
def query_for_answer(question: str, doc_type: str = "contract") -> str:
    retrieved_chunks = retrieve_and_rerank(question)
    context = "\n".join(retrieved_chunks)

    prompt = f"""
You are assisting with understanding a {doc_type}.
Question: {question}

Relevant context from the document:
{context}

Answer the question in plain English, clearly and concisely.
"""
    model = genai.GenerativeModel("gemini-1.5-flash")
    response = model.generate_content(prompt)
    return response.text

In [ ]:

def risk_check(doc_type: str = "contract") -> str:
    retrieved_chunks = retrieve_and_rerank("potential risks or penalties", top_k=8)
    context = "\n".join(retrieved_chunks)

    prompt = f"""
You are a legal risk detector. The user uploaded a {doc_type}.
Relevant document context:
{context}

Identify any clauses that may be risky or unfavorable (penalties, hidden fees, unilateral rights, etc.).
Return a short checklist in plain English.
If no risks are found, say: "No significant risks detected."
"""
    model = genai.GenerativeModel("gemini-1.5-flash")
    response = model.generate_content(prompt)
    return response.text

In [ ]:
# Ingest once
raw_chunks = ingest_document("/content/sample_rental_doc.pdf", doc_id="rental1", collection=collection)

# Retrieval-based (via vector DB)
print(simplify_clause("termination clause", doc_type="rental agreement"))
print(query_for_answer("Who is responsible for repairs?", doc_type="rental agreement"))
print(risk_check(doc_type="rental agreement"))




✅ Document rental1 ingested with 6 chunks.
Okay, let's break down your rental agreement's termination clause and related points into plain English.

---

### Termination Clause: Simple Explanation

This agreement can be ended early (before its official end date) by **either you (the Tenant) or the Landlord.**

To do so, the party wanting to end the agreement must give the other party **one month's written notice.**

---

**What else happens when the agreement ends (either early or on its scheduled date):**

*   **Your Security Deposit (Clause 6):**
    *   The Landlord must refund your security deposit when you hand over the property, after deducting any unpaid bills or costs for damages caused by your negligence (normal wear and tear and "acts of God" don't count).
    *   **Crucially, if the Landlord doesn't refund your security deposit on time, you have the right to stay in the property without paying rent or any other charges until they do.** This is in addition to any other ways y

In [ ]:
# Full summarization (via raw_chunks)
summaries = summarize_document(raw_chunks)
for title, bullets in summaries.items():
    print(f"\n📌 {title}")
    print(bullets)


📌 RENTAL AGREEMENT
- The Rental Agreement is made between the Owner, (Name of the Owner), and the Tenant, (Name of the Tenant), on (Date).
- The Owner owns the property located at (Complete Address of the Rented Property), as detailed in Annexure-I.
- The Tenant is renting the property for residential purposes.
- The rental includes parking for two-wheelers and four-wheelers.
- The agreement is subject to terms and conditions (specified elsewhere in the full agreement).


📌 NOW THIS DEED WITNESSETH AS FOLLOWS:
- Rent commences on the Starting Date of Agreement and runs until the Expiry Date of Agreement, with possible extension by mutual consent.  Monthly rent is Rs.(Amount of rent in Numbers), excluding utilities, payable by the 7th of each month.
- A monthly maintenance charge of Rs.(Amount in Numbers) is payable for generator, elevator, security, and common area upkeep.  Tenant pays separately for elevator and generator running costs, and all electricity and water bills during tena